# Wire OpenTelemetry traces to Application Insights and run groundedness and relevance evaluators over a RAG app

Your team shipped a RAG chatbot last Tuesday. By Thursday the product manager is asking "how do we know if it is any good?" and "why is page response taking 8 seconds sometimes?" You inherited a chatbot with no observability beyond stdout logs and a vague sense that the answers are usually fine. You have one session to instrument the app with OpenTelemetry, send the traces to Application Insights, run groundedness and relevance evaluators over a 20-prompt eval set you author, and find at least one bad trace so the team has a concrete remediation target by Monday's standup.

What you will build:

- A Foundry hub plus project in `eastus`, plus a `gpt-4o-mini` deployment.
- A Log Analytics workspace backing an Application Insights resource (Workspace-based App Insights is the only supported mode as of February 2024).
- A minimal RAG application wrapped in OpenTelemetry spans, traces flowing to Application Insights.
- The `GroundednessEvaluator` and `RelevanceEvaluator` from `azure-ai-evaluation` running over a 20-prompt eval set with reference answers.
- A KQL query in Log Analytics that finds the trace associated with a sub-3 groundedness response so the team has a concrete replay target.

**Time.** Plan for about 60 minutes hands-on. Foundry provisioning is 5 to 8 minutes; App Insights and Log Analytics provisioning is fast; the eval run is the slow part because 20 prompts pass through two evaluators each. Budget up to 100 minutes total.

**Cost.** This is the cheapest lab in the AI-103 track that does real work. Application Insights gives you 5 GB free ingestion per month; the eval evaluators cost cents per run; the lab volume is well under a megabyte. Expect $0.00 to $0.20 per session. Observability is the unsexy half of shipping production AI; the lab pays for itself by surfacing one bad trace you can actually fix.

This lab maps to AI-103 Domain 1: Plan and manage an Azure AI solution (28% of exam weight) and Domain 2: Implement generative AI and agentic solutions (33%).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Azure SDKs and the OpenTelemetry distro.
# - azure-monitor-opentelemetry-distro is the supported package as of
#   March 2026. The older opencensus packages are deprecated.
# - azure-ai-evaluation hosts the GroundednessEvaluator and
#   RelevanceEvaluator surfaces.
# - azure-monitor-query exposes the KQL client used to query Log Analytics.

!pip install --quiet labrun-checks[azure]==0.7.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0 azure-mgmt-applicationinsights>=4.0.0 azure-mgmt-loganalytics>=13.0.0 azure-monitor-opentelemetry-distro>=1.6.0 azure-monitor-query>=1.3.0 azure-ai-evaluation>=1.0.0 openai>=2.0.0 pandas>=2.0.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import getpass
import json
import time
from datetime import timedelta

import pandas as pd
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.core.exceptions import (
    HttpResponseError,
    ResourceNotFoundError,
    ClientAuthenticationError,
)
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from azure.mgmt.applicationinsights import ApplicationInsightsManagementClient
from azure.mgmt.applicationinsights.models import ApplicationInsightsComponent
from azure.mgmt.loganalytics import LogAnalyticsManagementClient
from azure.mgmt.loganalytics.models import Workspace, WorkspaceSku
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from openai import AzureOpenAI

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)
from labrun_checks.adapters.azure import AzureCleanupAdapter

LAB_ID = "evaluation-and-observability"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4omini"
LOG_ANALYTICS_NAME = f"labrun-{LAB_ID}-law"
APP_INSIGHTS_NAME = f"labrun-{LAB_ID}-ai"
MODEL_NAME = "gpt-4o-mini"
MODEL_VERSION = "2024-07-18"
MODEL_CAPACITY_TPM = 30

# Resolved during setup.
SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
AOAI_ACCOUNT_ENDPOINT = None
APP_INSIGHTS_CONNECTION_STRING = None
LOG_ANALYTICS_WORKSPACE_ID = None
APP_INSIGHTS_OPERATION_IDS = []
EVAL_RESULTS_DF = None
KQL_JOIN_RESULTS = None
TRACE_FOR_BAD_GROUNDEDNESS = None
AZURE_CREDENTIAL = None

# Pricing for wallet checks.
PRICE_PER_1M_INPUT_USD = 0.15
PRICE_PER_1M_OUTPUT_USD = 0.60

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials per
# LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. This lab pins {REGION}.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION, "resource_group": RESOURCE_GROUP}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# No critical resources: Application Insights ingestion is free under 5 GB
# per month; Log Analytics has no fixed monthly fee; GPT-4o-mini Standard
# does not bill at zero traffic. Cleanup is for orphan-scan hygiene and to
# prevent the App Insights resource from quietly accepting ingestion if the
# connection string is reused.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="aoai_deployment",
        id=DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {DEPLOYMENT_NAME}"
        ),
        extra={"account_name": AOAI_ACCOUNT_NAME},
    ),
    CleanupResource(
        type="application_insights",
        id=APP_INSIGHTS_NAME,
        region=REGION,
        cli_delete_command=(
            f"az monitor app-insights component delete "
            f"--app {APP_INSIGHTS_NAME} --resource-group {RESOURCE_GROUP}"
        ),
    ),
    CleanupResource(
        type="log_analytics_workspace",
        id=LOG_ANALYTICS_NAME,
        region=REGION,
        cli_delete_command=(
            f"az monitor log-analytics workspace delete "
            f"--workspace-name {LOG_ANALYTICS_NAME} --resource-group {RESOURCE_GROUP} "
            f"--yes --force"
        ),
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print(f"Run the cleanup cell first, or: az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Provision Foundry plus Application Insights backed by a Log Analytics workspace

Application Insights does not stand alone any more. Since February 2024, every new App Insights resource must point at a Log Analytics workspace via `workspace_resource_id`. The data lives in the workspace; the App Insights resource is a labeled view over it.

Build it in this order:

1. Create the resource group, hub, project, and `gpt-4o-mini` deployment. Same shape as Lab 01.
2. Create the Log Analytics workspace via `LogAnalyticsManagementClient.workspaces.begin_create_or_update` at the `PerGB2018` SKU (the pay-as-you-go default).
3. Create the Application Insights resource via `ApplicationInsightsManagementClient.components.begin_create_or_update` with `kind="web"`, `application_type="web"`, and `workspace_resource_id` set to the workspace's full ARM id. Wait for the LRO.
4. Read the connection string off the App Insights properties. Capture it in `APP_INSIGHTS_CONNECTION_STRING` for the OpenTelemetry distro.

**Trap to avoid: classic (non-workspace) Application Insights.** If your `components` create call does not pass `workspace_resource_id`, Azure rejects it. Older tutorials show the classic shape; ignore them.

In [ ]:
# NBVAL_SKIP
# Task 1: Provision Foundry, deployment, Log Analytics workspace, App Insights.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
law_client = LogAnalyticsManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ai_client = ApplicationInsightsManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
).result()

project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id},
}
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
project_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
).result()

for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
aoai_account = cs_client.accounts.get(RESOURCE_GROUP, AOAI_ACCOUNT_NAME)
AOAI_ACCOUNT_ENDPOINT = aoai_account.properties.endpoint
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")

deployment_payload = {
    "sku": {"name": "Standard", "capacity": MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": MODEL_NAME, "version": MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Watching the model deployment go through Succeeded purgatory...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, DEPLOYMENT_NAME, deployment_payload,
).result()
print(f"GPT-4o-mini deployment ready: {DEPLOYMENT_NAME}")

# Log Analytics workspace.
law_payload = Workspace(
    location=REGION,
    tags=lab_tags,
    sku=WorkspaceSku(name="PerGB2018"),
    retention_in_days=30,
)
print("Provisioning Log Analytics workspace, this is fast...")
law_workspace = law_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, LOG_ANALYTICS_NAME, law_payload,
).result()
LOG_ANALYTICS_WORKSPACE_ID = law_workspace.id
print(f"Log Analytics workspace ready: {LOG_ANALYTICS_NAME}")

# Application Insights resource (workspace-based mode is the only supported
# shape since February 2024). The workspace_resource_id is what binds the
# component to the Log Analytics data store.
ai_payload = ApplicationInsightsComponent(
    location=REGION,
    tags=lab_tags,
    kind="web",
    application_type="web",
    workspace_resource_id=LOG_ANALYTICS_WORKSPACE_ID,
)
print("Provisioning the Application Insights component, hold for a few seconds...")
# YOUR CODE: Create the App Insights component via
# ai_component = ai_client.components.begin_create_or_update(
#     RESOURCE_GROUP, APP_INSIGHTS_NAME, ai_payload,
# ).result()
# Store the result in ai_component.

APP_INSIGHTS_CONNECTION_STRING = ai_component.connection_string
print(f"Application Insights component ready: {APP_INSIGHTS_NAME}")
print(f"Connection string captured: {APP_INSIGHTS_CONNECTION_STRING[:60]}...")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Application Insights component exists, points at the
# Log Analytics workspace via workspace_resource_id, provisioningState is
# Succeeded, and connection_string is non-empty with an IngestionEndpoint
# token (the structural marker that distinguishes valid connection strings
# from old instrumentation-key-only formats).

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        aimgmt = ApplicationInsightsManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
        try:
            comp = aimgmt.components.get(RESOURCE_GROUP, APP_INSIGHTS_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Application Insights component {APP_INSIGHTS_NAME} does not exist "
                    f"in {RESOURCE_GROUP}. Did Task 1 finish?"
                ),
            )

        if (comp.location or "").lower() != REGION:
            return CheckpointResult(
                status="fail",
                error_reason=f"Component location is {comp.location!r}, expected {REGION!r}.",
            )

        ws_id = getattr(comp, "workspace_resource_id", None)
        if not ws_id:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Component has no workspace_resource_id. Classic (non-workspace) "
                    "Application Insights was retired in February 2024; pass "
                    "workspace_resource_id pointing at the Log Analytics workspace's ARM id."
                ),
            )
        if LOG_ANALYTICS_NAME.lower() not in ws_id.lower():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"workspace_resource_id {ws_id!r} does not reference the lab's "
                    f"Log Analytics workspace {LOG_ANALYTICS_NAME!r}."
                ),
            )

        prov = getattr(comp, "provisioning_state", None)
        if prov and prov.lower() != "succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=f"provisioning_state is {prov!r}, expected 'Succeeded'.",
            )

        cs = comp.connection_string or ""
        if "IngestionEndpoint" not in cs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"connection_string {cs!r:.80} does not contain 'IngestionEndpoint'. "
                    f"That token is the structural marker of a modern App Insights "
                    f"connection string."
                ),
            )

        tags = comp.tags or {}
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Component is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. Found: {tags}"
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint says which property is wrong. The most common miss: leaving `workspace_resource_id` unset on the payload, which causes Azure to reject the component with `WorkspaceResourceIdRequired`. The second most common: forgetting to call `.result()` on the poller and capturing the in-progress component before connection_string is populated.

</details>

<details><summary>Hint 2 (direction)</summary>

One LRO. `ai_component = ai_client.components.begin_create_or_update(RESOURCE_GROUP, APP_INSIGHTS_NAME, ai_payload).result()`. The `ai_payload` already has `kind="web"`, `application_type="web"`, and `workspace_resource_id=LOG_ANALYTICS_WORKSPACE_ID` set. After `.result()` the connection string is accessible via `ai_component.connection_string`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
ai_component = ai_client.components.begin_create_or_update(
    RESOURCE_GROUP, APP_INSIGHTS_NAME, ai_payload,
).result()
```

</details>

**Wallet check.** Still at $0.00. Log Analytics and App Insights are free at the volumes this lab generates (first 5 GB ingestion per month is free for App Insights; Log Analytics pay-as-you-go is gated by ingestion which is well under a megabyte for us). Coffee remains the day's biggest expense.

## Task 2: Configure the OpenTelemetry distro and emit a span to Application Insights

The Azure Monitor OpenTelemetry distro auto-instruments the Azure SDKs and exposes a clean span API. You point it at the connection string, wrap a chunk of code in a span, and the SDK ships it to Application Insights via the ingestion endpoint.

Build it in this order:

1. Call `configure_azure_monitor(connection_string=APP_INSIGHTS_CONNECTION_STRING)` from `azure.monitor.opentelemetry`.
2. Get a tracer: `tracer = trace.get_tracer("labrun.rag.chat")`.
3. Wrap a minimal RAG-style chat call in a span named `rag.chat`. Inside the span, call the `gpt-4o-mini` deployment with the user's prompt and a tiny canned retrieval context.
4. Set custom span attributes: `span.set_attribute("prompt", prompt)`, `span.set_attribute("response", reply)`, `span.set_attribute("groundedness_score", -1)` (a placeholder; the eval phase populates the real score in Task 3 via a separate span path).
5. Capture the OTel `operation_id` from the active span context. Append to `APP_INSIGHTS_OPERATION_IDS` so downstream cells can find the span via KQL.

The first span typically appears in Log Analytics in 30 to 90 seconds. The OBSERVE phase below polls until the span lands.

In [ ]:
# NBVAL_SKIP
# Task 2 (CODE phase): Configure the OpenTelemetry distro and emit a span.

from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace

token_provider = get_bearer_token_provider(
    AZURE_CREDENTIAL, "https://cognitiveservices.azure.com/.default",
)
aoai_client = AzureOpenAI(
    azure_endpoint=AOAI_ACCOUNT_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-08-01-preview",
)

# Tiny canned retrieval context for the first span. The full eval set runs
# in Task 3.
canned_context = (
    "To rotate the API key, sign in to the admin console, open Settings, "
    "select Security, and click Rotate Key. The current key remains valid "
    "for 24 hours so deployments have time to redeploy with the new value."
)
canned_prompt = "How do I rotate the API key?"

# YOUR CODE: Configure the Azure Monitor OpenTelemetry distro pointed at the
# connection string captured in Task 1:
# configure_azure_monitor(connection_string=APP_INSIGHTS_CONNECTION_STRING)

tracer = trace.get_tracer("labrun.rag.chat")

with tracer.start_as_current_span("rag.chat") as span:
    reply = aoai_client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": f"Answer using only this context:\n{canned_context}"},
            {"role": "user", "content": canned_prompt},
        ],
    ).choices[0].message.content

    span.set_attribute("prompt", canned_prompt)
    span.set_attribute("response", reply or "")
    op_id = trace.get_current_span().get_span_context().trace_id
    op_id_hex = format(op_id, "032x")
    APP_INSIGHTS_OPERATION_IDS.append(op_id_hex)

print(f"Span emitted. Operation id: {op_id_hex}")
print(f"Assistant reply: {reply}")
print()
print("Asking App Insights to wake up and ingest the span, gives it about 60 seconds...")

In [ ]:
# NBVAL_SKIP
# Task 2 (OBSERVE phase): Query Log Analytics with KQL to confirm the span
# landed. The validator runs this same path; this cell prints the count and
# operation_id so the student can drill in via the Azure portal if needed.

logs_client = LogsQueryClient(AZURE_CREDENTIAL)

kql = (
    "AppDependencies "
    "| where Name == 'rag.chat' "
    "| where TimeGenerated > ago(5m) "
    "| project TimeGenerated, OperationId, Name, DurationMs, Properties "
    "| limit 10"
)
print("Querying Log Analytics for the rag.chat span, retrying for up to 3 minutes...")

found = False
for attempt in range(6):
    response = logs_client.query_workspace(
        workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
        query=kql,
        timespan=timedelta(minutes=10),
    )
    if response.status == LogsQueryStatus.SUCCESS:
        rows = response.tables[0].rows if response.tables else []
        if rows:
            print(f"Found {len(rows)} matching span(s) in Log Analytics.")
            for r in rows[:3]:
                print(f"  TimeGenerated={r[0]}  OperationId={r[1]}  DurationMs={r[3]}")
            found = True
            break
    print(f"  Attempt {attempt + 1}: no spans yet, waiting 30 seconds...")
    time.sleep(30)

if not found:
    print("Ingestion lag is unusually long. The validator will retry once more.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: At least one rag.chat span landed in Log Analytics within the
# last 10 minutes and its OperationId matches one we captured. Retries up to
# 3 minutes since App Insights ingestion has a 30-90s lag.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        if not APP_INSIGHTS_OPERATION_IDS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "APP_INSIGHTS_OPERATION_IDS is empty. Did Task 2 emit the rag.chat span?"
                ),
            )
        if not LOG_ANALYTICS_WORKSPACE_ID:
            return CheckpointResult(
                status="fail",
                error_reason="LOG_ANALYTICS_WORKSPACE_ID is unset. Re-run Task 1.",
            )

        client_local = LogsQueryClient(AZURE_CREDENTIAL)
        kql_local = (
            "AppDependencies "
            "| where Name == 'rag.chat' "
            "| where TimeGenerated > ago(10m) "
            "| project OperationId "
            "| limit 50"
        )

        for attempt in range(6):
            resp = client_local.query_workspace(
                workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
                query=kql_local,
                timespan=timedelta(minutes=15),
            )
            if resp.status == LogsQueryStatus.SUCCESS and resp.tables:
                op_ids_seen = {str(r[0]).replace("-", "").lower() for r in resp.tables[0].rows}
                captured = {x.lower() for x in APP_INSIGHTS_OPERATION_IDS}
                if op_ids_seen & captured:
                    return CheckpointResult(status="pass")
            time.sleep(30)

        return CheckpointResult(
            status="fail",
            error_reason=(
                "No matching rag.chat span landed in Log Analytics within 3 minutes. "
                "Ingestion lag is usually 30 to 90s; if it persists, confirm "
                "configure_azure_monitor was called with the captured connection string."
            ),
        )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

If the span never lands, your configure_azure_monitor call ran with a wrong connection string, or it never ran. If the span lands but operation_ids do not match, the format conversion is off; OpenTelemetry's trace_id is a 128-bit int rendered as 32-char hex with no dashes.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `configure_azure_monitor(connection_string=APP_INSIGHTS_CONNECTION_STRING)`. The connection string is already captured. After this call, any tracer obtained via `trace.get_tracer(...)` ships spans to App Insights automatically.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (CODE phase):

```python
configure_azure_monitor(connection_string=APP_INSIGHTS_CONNECTION_STRING)
```

</details>

**Wallet check.** The chat call cost a fraction of a cent. App Insights ingestion of one span is free under the 5 GB monthly tier. Coffee continues to dominate.

## Task 3: Run GroundednessEvaluator and RelevanceEvaluator over a 20-prompt eval set

The eval set is the thing the team can show the product manager. Each row has a query, a retrieved context, a generated response, and a hand-authored ground-truth answer. The evaluators score every row on a 1 to 5 scale for groundedness (does the response come from the context?) and relevance (does the response answer the query?).

Build it in this order:

1. Load the seeded 20-row eval set provided in the cell below. 15 rows are well-grounded; 5 rows are deliberately ungrounded (the response includes claims not in the context) so the evaluators can find them.
2. Instantiate `GroundednessEvaluator` and `RelevanceEvaluator` from `azure-ai-evaluation` with `model_config` pointing at the GPT-4o-mini deployment (the evaluators use an LLM judge under the hood).
3. Loop over every row, call each evaluator, capture the per-row scores. Store the result as a Pandas DataFrame in `EVAL_RESULTS_DF`.
4. Wrap every model call (the RAG chat AND the evaluator call) in OpenTelemetry spans tagged with the row's `query` so the OBSERVE-phase KQL query can join scores back to spans.

**Score scale.** Both evaluators return 1 to 5 (not 0 to 1). The validator asserts at least one row has `groundedness < 3` so the OBSERVE phase has a concrete bad-trace target to find.

In [ ]:
# NBVAL_SKIP
# Task 3 (CODE phase): Run groundedness and relevance evaluators over the
# 20-row eval set with span-tagging.

from azure.ai.evaluation import GroundednessEvaluator, RelevanceEvaluator

# 20-row eval set. 15 well-grounded rows; 5 deliberately ungrounded so the
# evaluators have something to flag.
EVAL_SET = []
for i in range(15):
    EVAL_SET.append({
        "query": f"What is configuration option {i}?",
        "context": (
            f"Section {i // 5 + 1}: configuration option {i} controls the corresponding "
            f"feature. Default value is enabled."
        ),
        "response": f"Configuration option {i} controls the corresponding feature; default is enabled.",
        "ground_truth": f"Configuration option {i} is enabled by default.",
    })
for i in range(15, 20):
    # Ungrounded rows: response invents a claim not supported by the context.
    EVAL_SET.append({
        "query": f"What is configuration option {i}?",
        "context": (
            f"Section {i // 5 + 1}: configuration option {i} controls the corresponding "
            f"feature. Default value is enabled."
        ),
        "response": (
            f"Configuration option {i} is disabled by default and requires a paid "
            f"subscription tier to enable. The minimum activation cost is $99 per month."
        ),
        "ground_truth": f"Configuration option {i} is enabled by default.",
    })

print(f"Eval set seeded: {len(EVAL_SET)} rows (15 grounded, 5 ungrounded).")

model_config = {
    "azure_endpoint": AOAI_ACCOUNT_ENDPOINT,
    "azure_deployment": DEPLOYMENT_NAME,
    "api_version": "2024-08-01-preview",
}

# YOUR CODE: Instantiate the evaluators. Pass model_config to each:
# groundedness_eval = GroundednessEvaluator(model_config=model_config)
# relevance_eval = RelevanceEvaluator(model_config=model_config)

print("Running the groundedness evaluator over 20 prompts, this is the slow part of the lab...")

rows = []
for row in EVAL_SET:
    with tracer.start_as_current_span("rag.eval") as span:
        span.set_attribute("eval.query", row["query"])
        try:
            g = groundedness_eval(
                query=row["query"], context=row["context"], response=row["response"],
            )
        except Exception as exc:
            g = {"groundedness": None, "groundedness_reason": f"eval_error: {exc}"}
        try:
            r = relevance_eval(
                query=row["query"], response=row["response"], context=row["context"],
            )
        except Exception as exc:
            r = {"relevance": None, "relevance_reason": f"eval_error: {exc}"}

        g_score = g.get("groundedness")
        r_score = r.get("relevance")
        span.set_attribute("eval.groundedness", g_score if g_score is not None else -1)
        span.set_attribute("eval.relevance", r_score if r_score is not None else -1)
        rows.append({
            "query": row["query"],
            "response": row["response"],
            "groundedness": g_score,
            "relevance": r_score,
            "operation_id": format(trace.get_current_span().get_span_context().trace_id, "032x"),
        })

EVAL_RESULTS_DF = pd.DataFrame(rows)
print()
print(EVAL_RESULTS_DF[["query", "groundedness", "relevance"]].head(10).to_string(index=False))
print()
print(f"Rows with groundedness < 3: {(EVAL_RESULTS_DF['groundedness'] < 3).sum()}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: 20 rows in, 20 rows out; every row has a numeric
# groundedness in 1-5 and a numeric relevance in 1-5; at least one row has
# groundedness < 3 (so Task 4 has a bad trace to find).

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        if EVAL_RESULTS_DF is None or len(EVAL_RESULTS_DF) != 20:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EVAL_RESULTS_DF has "
                    f"{len(EVAL_RESULTS_DF) if EVAL_RESULTS_DF is not None else 'no'} rows, "
                    f"expected 20."
                ),
            )

        for col in ("groundedness", "relevance"):
            if col not in EVAL_RESULTS_DF.columns:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"EVAL_RESULTS_DF is missing column {col!r}.",
                )
            for i, value in enumerate(EVAL_RESULTS_DF[col].tolist()):
                if value is None or (isinstance(value, float) and pd.isna(value)):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Row {i} {col} is None or NaN. The evaluator failed on that row; "
                            f"check the model_config and retry."
                        ),
                    )
                try:
                    fv = float(value)
                except (TypeError, ValueError):
                    return CheckpointResult(
                        status="fail",
                        error_reason=f"Row {i} {col} is not numeric: {value!r}.",
                    )
                if fv < 1 or fv > 5:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Row {i} {col}={fv} is outside the 1-5 range. The Foundry "
                            f"evaluators emit on a 1-5 scale; if you see 0-1, check that "
                            f"you instantiated GroundednessEvaluator / RelevanceEvaluator "
                            f"(not a custom 0-1 reimplementation)."
                        ),
                    )

        ungrounded = (EVAL_RESULTS_DF["groundedness"] < 3).sum()
        if ungrounded < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No rows with groundedness < 3 in the eval result. The seeded set has "
                    f"5 deliberately ungrounded rows; the evaluator should flag at least 1. "
                    f"Re-run; LLM-as-judge can occasionally score these higher than expected."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

If rows have None scores, your `model_config` is missing or the evaluator hit a quota error. If groundedness is always above 3, the evaluator did not flag the seeded ungrounded rows; re-run, LLM-judge is non-deterministic and occasionally lenient on borderline rows. If the score scale is 0 to 1 instead of 1 to 5, you instantiated a different evaluator.

</details>

<details><summary>Hint 2 (direction)</summary>

Two constructors: `groundedness_eval = GroundednessEvaluator(model_config=model_config)` and `relevance_eval = RelevanceEvaluator(model_config=model_config)`. The `model_config` dict is already prepared. The evaluators are callables; pass `query=`, `response=`, and `context=` as keyword arguments to each.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
groundedness_eval = GroundednessEvaluator(model_config=model_config)
relevance_eval = RelevanceEvaluator(model_config=model_config)
```

</details>

**Wallet check.** Twenty rows times two evaluators each calls the model 40 times. At ~700 input plus ~150 output tokens per call, that totals ~28,000 input plus ~6,000 output = $0.0042 plus $0.0036 = under one cent. The eval is the most expensive single block in the lab and it still rounds to nothing.

## Task 4: Join the eval scores back to App Insights traces via KQL and find the bad trace

The product manager wants a concrete trace to replay. You have evaluator scores in a DataFrame and OTel spans in App Insights, joined by `operation_id`. The KQL query pulls every `rag.eval` span from the last 30 minutes whose `eval.groundedness` attribute is below 3, then joins on the `EVAL_RESULTS_DF` operation_ids.

Build it in this order:

1. Compose a KQL query that filters `AppDependencies` by `Name == 'rag.eval'`, parses the `Properties` column (App Insights stores OTel attributes there as JSON), and projects rows where the groundedness attribute is below 3.
2. Execute via `LogsQueryClient.query_workspace`.
3. Capture at least one row in `KQL_JOIN_RESULTS` and pick one trace `operation_id` to surface as `TRACE_FOR_BAD_GROUNDEDNESS`. This is what the team replays.

**Where attributes live in KQL.** OTel custom attributes land under `Properties` (a dynamic JSON object) on the `AppDependencies` table. `parse_json(Properties).["eval.groundedness"]` is the right reach; a bare column name returns null.

In [ ]:
# NBVAL_SKIP
# Task 4 (OBSERVE phase): KQL join to find sub-3 groundedness traces.

print("Querying Application Insights with KQL for sub-threshold groundedness scores...")

kql_join = (
    "AppDependencies "
    "| where Name == 'rag.eval' "
    "| where TimeGenerated > ago(30m) "
    "| extend props = parse_json(Properties) "
    "| extend groundedness = todouble(props['eval.groundedness']) "
    "| extend relevance = todouble(props['eval.relevance']) "
    "| extend query_text = tostring(props['eval.query']) "
    "| where groundedness >= 1 and groundedness < 3 "
    "| project TimeGenerated, OperationId, query_text, groundedness, relevance "
    "| limit 25"
)

# Retry to absorb the 30-90s ingestion lag.
KQL_JOIN_RESULTS = None
for attempt in range(6):
    # YOUR CODE: Run the KQL query via
    # response = logs_client.query_workspace(
    #     workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
    #     query=kql_join,
    #     timespan=timedelta(minutes=45),
    # )
    if response.status == LogsQueryStatus.SUCCESS and response.tables:
        rows = response.tables[0].rows
        if rows:
            KQL_JOIN_RESULTS = rows
            break
    print(f"  Attempt {attempt + 1}: no rows yet, waiting 30 seconds...")
    time.sleep(30)

if KQL_JOIN_RESULTS:
    print(f"Found {len(KQL_JOIN_RESULTS)} sub-3 groundedness trace(s).")
    for r in KQL_JOIN_RESULTS[:3]:
        print(f"  OperationId={r[1]}  groundedness={r[3]}  query={r[2]!r:.80}")
    TRACE_FOR_BAD_GROUNDEDNESS = str(KQL_JOIN_RESULTS[0][1])
    print()
    print(f"Trace to replay on Monday: {TRACE_FOR_BAD_GROUNDEDNESS}")
else:
    print("No sub-3 groundedness traces found in the KQL window. Re-run Task 3.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: KQL_JOIN_RESULTS has at least one row; the picked
# TRACE_FOR_BAD_GROUNDEDNESS operation_id matches an operation_id in
# EVAL_RESULTS_DF whose groundedness is < 3. This proves the join worked.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        if not KQL_JOIN_RESULTS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "KQL_JOIN_RESULTS is empty or None. Either the KQL query did not run, "
                    "the parse_json(Properties) expression is mistyped, or the span "
                    "ingestion lag exceeded 3 minutes."
                ),
            )
        if not TRACE_FOR_BAD_GROUNDEDNESS:
            return CheckpointResult(
                status="fail",
                error_reason="TRACE_FOR_BAD_GROUNDEDNESS is unset; pick KQL_JOIN_RESULTS[0][1].",
            )

        if EVAL_RESULTS_DF is None:
            return CheckpointResult(
                status="fail",
                error_reason="EVAL_RESULTS_DF is None. Re-run Task 3.",
            )
        bad_ops = EVAL_RESULTS_DF[EVAL_RESULTS_DF["groundedness"] < 3]["operation_id"].tolist()
        bad_ops_normalised = {str(x).replace("-", "").lower() for x in bad_ops}
        picked = str(TRACE_FOR_BAD_GROUNDEDNESS).replace("-", "").lower()
        if picked not in bad_ops_normalised:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"TRACE_FOR_BAD_GROUNDEDNESS={TRACE_FOR_BAD_GROUNDEDNESS!r} does not match "
                    f"any sub-3 operation_id in EVAL_RESULTS_DF. The KQL filter or the "
                    f"join key is off; check that you compared eval.groundedness, not "
                    f"a different attribute name."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If KQL returns no rows, the `Properties` column needs `parse_json` before you can index attributes. A bare `Properties.["eval.groundedness"]` returns null. If KQL returns rows but the join in Checkpoint 4 fails, double-check the operation_id normalisation (KQL returns dashed UUIDs; the captured ones are hex).

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `response = logs_client.query_workspace(workspace_id=LOG_ANALYTICS_WORKSPACE_ID, query=kql_join, timespan=timedelta(minutes=45))`. The KQL string is already composed in `kql_join`. The timespan is the rolling window; 45 minutes absorbs the ingestion lag with slack.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
response = logs_client.query_workspace(
    workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
    query=kql_join,
    timespan=timedelta(minutes=45),
)
```

</details>

**Wallet check.** KQL queries are free at this volume (the 5 GB ingestion tier covers analysis too). Total session damage: under one cent. Coffee is comfortably 50x ahead.

## Cleanup

Time to tear it all down. The cell below runs the manifest in reverse-creation order: deployment, App Insights component, Log Analytics workspace (with `--force` to bypass the 14-day soft-delete window), project, hub, resource group. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST.

import sys

for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

# Patch resource.extra with constants resolved during the lab so the
# Azure adapter sees account_name, service_name, and project_endpoint
# at cleanup time (manifest is built before those constants are set).
for r in CLEANUP_MANIFEST:
    if r.type in ("aoai_deployment", "aoai_rai_policy") and AOAI_ACCOUNT_NAME:
        r.extra = {"account_name": AOAI_ACCOUNT_NAME}
    elif r.type == "search_index" and "SEARCH_SERVICE_NAME" in globals() and SEARCH_SERVICE_NAME:
        r.extra = {"service_name": SEARCH_SERVICE_NAME}
    elif r.type == "foundry_agent" and "PROJECT_ENDPOINT" in globals() and PROJECT_ENDPOINT:
        r.extra = {"project_endpoint": PROJECT_ENDPOINT}

result = run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: under one cent in a clean run, up to about twenty cents if you re-ran the eval set many times.** Application Insights and Log Analytics stay free under the 5 GB monthly tier. The Foundry hub, project, and Standard deployment carry no hourly fee at zero traffic; the resource group delete is the safety net. Check Azure Cost Management in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. Groundedness scored low on a response that nonetheless looked plausible to a human reader. Walk through what groundedness measures specifically, why a low-groundedness response can still look right to a human, and what additional evaluator you would add to catch the inverse failure (a well-grounded response to the wrong question).

2. The eval set was 20 prompts the team authored by hand. In production with 100,000 conversations per day, you cannot hand-author an eval set that covers the live distribution. Walk through how you would sample, anonymize, and label production traffic to build a maintainable eval set that catches groundedness drift over time, and which Azure surfaces (App Insights, Foundry datasets, Data Factory) you would wire together to do it.

## Exam-Style Practice

**Question 1 (Domain 1, evaluator selection):**

A team running a RAG-based customer-support chatbot has two failure modes they want to detect via evaluators: (a) the chatbot returns answers that are not supported by the retrieved chunks, and (b) the chatbot's retrieved chunks are not on-topic for the user's question. Which evaluator combination addresses both failure modes?

A. `ContentSafetyEvaluator` for (a) and `RelevanceEvaluator` for (b).

B. `GroundednessEvaluator` for (a) and `RelevanceEvaluator` for (b).

C. `RelevanceEvaluator` for both, with different threshold values.

D. `CoherenceEvaluator` for (a) and `FluencyEvaluator` for (b).

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: `ContentSafetyEvaluator` measures safety severity, not factual support.
- B is correct: `GroundednessEvaluator` measures whether the response is supported by the context (failure mode a). `RelevanceEvaluator` measures whether the response and context are relevant to the user's question (failure mode b). This is the documented pair for RAG quality evaluation in AI-103.
- C is partially right that relevance covers (b) but it does not cover the "supported by retrieved chunks" half.
- D is wrong: coherence and fluency measure language quality, not factual or topical correctness.

</details>

**Question 2 (Domain 2, observability architecture):**

A production agentic application emits OpenTelemetry traces. The team wants to query the traces to find spans where the agent's tool-call duration exceeded 2 seconds in the last hour. Which Azure surface and query are the right tool?

A. Azure Monitor Metrics; query for `tool_call_duration` with a 2-second threshold and a 1-hour window.

B. Application Insights Live Metrics Stream; watch for slow spans in real time.

C. Application Insights / Log Analytics with KQL: `AppDependencies | where TimeGenerated > ago(1h) | where Name contains "tool" | where DurationMs > 2000 | project OperationId, Name, DurationMs`.

D. Azure Service Bus dead-letter queue; long-running tool calls are sent there automatically.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: Azure Monitor Metrics handles aggregated numeric metrics, not per-span trace data.
- B is wrong: Live Metrics is a real-time view, not a historical query. The team wants "last hour" which is a historical query.
- C is correct: the canonical AI-103 pattern. Traces sent via OTel land in Log Analytics (under AppDependencies for dependency spans); KQL is the query language. DurationMs is in milliseconds.
- D is wrong: Service Bus is unrelated. There is no auto-routing of long-running OTel spans to a dead-letter queue.

</details>